# The MediaWiki API

Some written sources are hosted on instances of [MediaWiki](https://www.mediawiki.org/), the most widespread wiki platform, developed by [the Wikimedia movement](https://www.wikimedia.org/) with the support of [the Wikimedia Foundation](https://wikimediafoundation.org/) and best known from [Wikipedia](https://www.wikipedia.org/). Notable instances are [Wiktionary](https://www.wiktionary.org/), [WikiSource](https://wikisource.org/), and [Heimskringla](https://heimskringla.no/).

A number of APIs are available for accessing these resources. There is a purpose-built [Wikipedia API](https://pypi.org/project/Wikipedia-API/), but it provides access to Wikipedia only and it is of no use in accessing other wikis. So instead we'll use [`pymediawiki`](https://pypi.org/project/pymediawiki/), which takes an API URL as an argument, or defaults to Wikipedia if no URL is provided. (Please note that `pymediawiki` is not available through `conda`, so install it with `pip install pymediawiki`.) 

There are nevertheless a few challenges to do with document structure. This may be demonstrated by demonstrating access to the rigorously structured Wikipedia first, then moving on to other resources.

## Wikipedia

In [1]:
from mediawiki import MediaWiki

Instantiate the `Mediawiki()` class with no `url` argument to gain access to Wikipedia. If you want a language edition other than English, set the `language` function with a string like `de` for German.

In [2]:
wiki = MediaWiki()
#wiki.language='de'

We can use the API just as we would the wiki's website, using functions like `search()`:

In [3]:
wiki.search('Battle of Maldon')

['Battle of Maldon',
 'The Battle of Maldon',
 'Maldon',
 'Balrog',
 'Heybridge, Maldon',
 'Maldon District',
 'Northern courage in Middle-earth',
 'Weapons and armour in Anglo-Saxon England',
 "The Homecoming of Beorhtnoth Beorhthelm's Son",
 'Fëanor']

We can then access a specific page using `page()`:

In [4]:
page = wiki.page('The Battle of Maldon')

And here is where we get into document structure. We normally expect every wiki page to have sections, which can be listed using `sections()` and accessed using `section()`:

In [5]:
page.sections

['The poem "The Battle of Maldon"',
 'Other sources',
 'Chronology',
 'Topography',
 'Manuscript sources',
 'See also',
 'References',
 'Bibliography',
 'Editions and translations',
 'External links']

In [6]:
page.section('Manuscript sources')

'In the Cotton library, the "Battle of Maldon" text had been in Otho A xii. The Elphinstone transcription is in the Bodleian Library, where it is pp. 7–12 of MS Rawlinson B. 203.'

Or we can access the full page using `content`:

In [7]:
doc = page.content
doc[:200]

'The Battle of Maldon took place on 10 or 11 August 991 AD near Maldon beside the River Blackwater in Essex, England, during the reign of Æthelred the Unready. Earl Byrhtnoth and his thegns led the Eng'

## Primary Source Wikis

And this is where some other resources fall short. As mentioned above, [WikiSource](https://wikisource.org/) is a major host of documents in the public domain, not unlike [Project Gutenberg](https://gutenberg.org/) (for which [a separate Python API wrapper is available](https://pypi.org/project/py-gutenberg/)), and like Wikipedia, it has language editions as well. A massive MediaWiki-hosted resource for out-of-copyright Old Norse text editions is [heimskringla.no](https://heimskringla.no/). But the document structure of neither can be accessed using the above methods, as will be demonstrated below.

We may start by loading the German-language Wikisource, and accessing the Old Low German _Heliand_:

In [8]:
wiki = MediaWiki(url='https://secure.wikimedia.org/wikisource/de/w/api.php')
page = wiki.page("Heliand")

Here our `sections` command yields an empty list of results, even though the wiki page clearly has sections:

In [9]:
page.sections

[]

With an empty list of sections, we are limited to calling the document as a whole. Let's try the `content` accessor for plaintext access:

In [10]:
page.content

''

Hmm, either these documents do not conform to the MediaWiki structure, or else the API falls short in some way. That leaves us with two options, both involving a degree of markup. Either we access the document as `wikitext`, which retains MediaWiki's markup for things like headings and links, or we use the `html` object and process the output using `BeautifulSoup`, the go-to HTML library.

In [11]:
wikitext = str(page.wikitext).split('\n')
wikitext[:30]

['{{Ohne Quelle}}',
 '',
 '{{Textdaten',
 '|VORIGER=',
 '|NÄCHSTER=',
 '|AUTOR=',
 '|ILLUSTRATOR=',
 '|TITEL=Heliand',
 '|SUBTITEL=',
 '|HERKUNFT=',
 '|HERAUSGEBER=',
 '|AUFLAGE=',
 '|VERLAG=',
 '|DRUCKER=',
 '|ENTSTEHUNGSJAHR=um 830',
 '|ERSCHEINUNGSJAHR=',
 '|ERSCHEINUNGSORT=',
 '|ÜBERSETZER=',
 '|ORIGINALTITEL=',
 '|ORIGINALSUBTITEL=',
 '|ORIGINALHERKUNFT=',
 '|WIKIPEDIA=Heliand',
 '|GND=',
 '|BILD=',
 '|QUELLE=',
 '|KURZBESCHREIBUNG=',
 '|SONSTIGES=',
 '|BENUTZERHILFE=',
 '|INDEXSEITE=',
 '|BEARBEITUNGSSTAND=unkorrigiert']

But HTML will be more practical, since we can use the `BeautifulSoup` library to parse it:

In [12]:
from bs4 import BeautifulSoup
html = page.html
soup = BeautifulSoup(html, 'html.parser')

Now we should be able to access HTML headings to access sections, even if we could not access them using the library's stock accessors. To find the correct information, inspect the HTML source either by outputting it in your notebook, or using the inspect source function in your browser (Ctrl+U, or Command+Option+U on macOS). We can then use the `find` method with an element name as the positional element, and something like `class_` for its name:

In [13]:
poem = soup.find('div', class_='poem')

Now we can output the content of our desired section using the `get_text()` accessor:

In [14]:
text = poem.get_text()
print(str(text)[:1200])


1 *Manega uuâron, · the sia iro môd gespôn, 
2 ........................., · that sia [bigunnun uuord godes], 
3 [reckean] that girûni, · that [thie] rîceo Crist 
4 undar mancunnea · mâriða gifrumida 
5 mid uuordun endi mid uuercun. · That uuolda thô uuîsara filo 
6 liudo barno loƀon, · lêra Cristes, 
7 hêlag uuord godas, · endi mid iro handon scrîƀan 
8 [berehtlîco] an buok, · huô sia [is gibodscip scoldin] 
9 frummian, firiho barn. · Than uuârun thoh sia fiori te thiu 
10 under thera menigo, · thia habdon maht godes, 
11 helpa fan himila, · hêlagna gêst, 
12 craft fan [Criste,] — · sia uurðun gicorana te thio, 
13 that sie than êuangelium · [ênan] [scoldun] 
14 an buok scrîƀan · endo sô manag gibod godes, 
15 hêlag himilisc uuord · sia ne muosta heliðo than mêr, 
16 firiho barno frummian, · neuan that sia fiori te thio 
17 thuru craft godas · gecorana uurðun, 
18 Matheus endi Marcus, · — sô uuârun thia man hêtana — 
19 [Lucas endi Iohannes]; · sia uuârun [gode] [lieƀa], 
20 uuirðiga 

From that point, we can process the output just [as we would](https://github.com/langeslag/ehtc/blob/main/demo/load_plaintext.ipynb) a plaintext corpus: split into lines, tokenize, normalize, etc.

Let's try that exercise again, now using Heimskringla:

In [15]:
wiki = MediaWiki(url='http://heimskringla.no/api.php')

In [16]:
wiki.search('Stjórn')

['Stjórn', 'Fortale (Stjórn)', 'Prolog (Stjórn)', 'Indledning (Stjórn)']

In [17]:
page = wiki.page('Stjórn')

With this resource, if we make `sections()` or `content` calls we actually get an error; apparently, MediaWiki was installed on the server without the requisite extensions. So let's go the BeautifulSoup way here as well.

Same as before, we just want the text itself, not the acknowledgements at the top. [The webpage](https://heimskringla.no/wiki/Stj%C3%B3rn) actually shows that this is a table of contents document, with links to individual pages. So to load the Book of Joshua, one constituent part of _Stjórn_, we could try the following:

In [18]:
page = wiki.page('Josvæ Bog')
html = page.html
soup = BeautifulSoup(html, 'html.parser')
paragraphs = []
for paragraph in soup.find_all('p'):
    text = paragraph.get_text()
    if text != '\n':
        paragraphs.append(str(text))
print(paragraphs[0])

146. Moyses agietr guds vin aðr hann andadizt hafðe at guds raaðe ok fyrersỏgn sett þann mann hertogha yfer allan Gyðinga lyð er heet Josue. faðer hans er nefndr Nun. Josue heet oðro nafne Jesus sun Naue. Ok efter andlaat Moysi vittraðiz gud drottinn Josue sua seghiande. Moyses minn þionn er andaðr. nu ris þu vpp ok flyt allan minn lyð yfer aana Jordan. aa þa iorð er ek man gefa Gydinga lyð. Huern þann stad sem þer komit fotum a man ek selia vnder ydart valld. sem ek sagða Moysi. fra eyðimork ok fialli Libano allt til hinnar miklu aar Eyfraaten. alla iorð þeirra þioða er Ethei heita til hafsins mikla moti solar setri. þetta er vmmerki ydars rikiss. Eingi maðr skal yðr megha i moti standa sua leingi sem þu lifer. Sua sem ek var meðr Moysi meðan hann lifdi. slikt hit sama skal ek nu meðr þer vera ok eigi þik vpp gefa ok eigi forlaata. Styrktz þu ok ver oflugr. þuiat þu skallt skipta iorðu þeirri medr þessum lyð. er ek heet[2] at gefa fedrum þinum. Styrktz þu ok ver stoðugr at fremia ok f

Or perhaps you can come up with a `for` loop that identifies each link in the table of contents, then processes the linked page in turn. That would be helpful for [the WikiSource text of Ælfric's _Lives of Saints_](https://en.wikisource.org/wiki/%C3%86lfric%27s_Lives_of_Saints) as well, where the chapters are actually contained in a subfolder. But if that's the edition you want, don't bother, unless you want the Modern English translation specifically --- you can get the Old English text through [YCOE](https://github.com/langeslag/ehtc/blob/main/demo/ycoe.ipynb) more easily and with full part-of-speech annotation and syntactical bracketing.